# Welcome to NER Finetuning with HF Transformers 

First of all, we need to import all necessary libraries and modules.

## Contents 
1. Configurations (Resources, Training, Optimization, Evaluation, Saving results)
2. Prepare training & dataset
3. Train (Finetune) a NER Model
4. Optimization
5. Evaluation

In [1]:
# import modules
import config
import train
import eval_opt

/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configurations

In this section, we set all of the configurations for the following steps (training, optimization, evaluation). Depending on which parts you want to run and which to leave out, corresponding config parts can be skipped if not needed.

### Resources

We need to state which resources we want to use for NER finetuning, namely the `dataset` and `model`. Therefore, we use the `Resource` class and define the `path` (either to a HF or a local directory) and `source` (`hf` or `local`) of our resources. By applying `set_name()`, the model or dataset name gets set automatically, otherwise the `model.name` can also be updated manually.

### Load Model

You can choose e.g. one of the following models and add them as `path` in the model config:
* FacebookAI/roberta-base
* dbmdz/bert-base-german-cased
* dbmdz/bert-base-historic-multilingual-cased
* flair/ner-german
* distilbert/distilbert-base-uncased

In [2]:
model = config.Resource(path="FacebookAI/roberta-base", source="hf")
model.set_name()
print(model.info())

roberta-base will be loaded from FacebookAI/roberta-base (via hf).


### Load Dataset

You can choose e.g. one of the following dataset configurations:
* `path="data/zefys2025.hf", source="local"`
* `path="eriktks/conll2003", source="hf"`
* `path="GermanEval/germeval_14", source="hf"`
* `path="data/hisgermaner.hf", source="local"`
* `path="data/newseye.hf", source="local"`

In [3]:
dataset = config.Resource(path="data/zefys2025.hf", source="local")
dataset.set_name()
print(dataset.info())

zefys2025 will be loaded from data/zefys2025.hf (via local).


In [4]:
train.set_torch_device()

device(type='cuda')

## 2. Prepare dataset

In [5]:
ner_dataset = train.load_ner_dataset(dataset.path, dataset.source)
ner_dataset["train"][1]

{'id': 10170,
 'tokens': ['"',
  'Die',
  '"',
  'Regierung',
  'ohne',
  '"',
  'Mehrheit',
  '"',
  '"',
  '"',
  'werde',
  'im',
  'Parlament',
  'die',
  'Antwort',
  'erhalten',
  ',',
  'die',
  'nur',
  'lauten',
  'könne',
  ':',
  '"',
  'Fort',
  'mit',
  'der',
  'Regierung',
  ',',
  'der',
  'schwarzweißroten',
  '"',
  'Flaggenverordnung',
  '!'],
 'ner_tags': [3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3,
  3]}

In [64]:
"""
# for hipe2020
from datasets import load_dataset
ner_dataset = load_dataset('bigscience-historical-texts/HIPE2020_sent-split', "de")
ner_dataset["train"][1]
"""

'\n# for hipe2020\nfrom datasets import load_dataset\nner_dataset = load_dataset(\'bigscience-historical-texts/HIPE2020_sent-split\', "de")\nner_dataset["train"][1]\n'

In [8]:
"""
# for roberta
import torch
from datetime import datetime
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification
import eval_opt
import evaluate
import numpy as np

def get_tokenizer(model_path):
    tokenizer = AutoTokenizer.from_pretrained(model_path, add_prefix_space=True)
    return tokenizer

tokenizer = get_tokenizer(model.path)
"""

In [6]:
tokenizer = train.get_tokenizer(model.path)

In [9]:
tokenized_dataset = train.prepare_dataset(ner_dataset, tokenizer)
label_list = train.get_label_list(ner_dataset)

Map: 100%|████████████████████████████████████████████████████████████████| 1635/1635 [00:00<00:00, 12324.15 examples/s]


In [10]:
label_list

['B-PER', 'B-ORG', 'I-LOC', 'O', 'I-PER', 'B-LOC', 'I-ORG']

## 3. Train (Finetune) a NER Model

To finetune our model for NER, we first need to define our `ner` task (see train.py). 

### Training Parameters

You can define a variable for the training parameters based on our default `TrainingParams` config and try to improve results using evaluation/optimization techniques, or start by overwriting the parameters for training manually. Training will only be performed once on the training dataset and evaluated once on the validation dataset. For more sophisticated methods, you can additionally run the Optimization and Evaluation parts to find ideal hyperparameter combinations and perform a more reliable evaluation.

In [11]:
training_params = config.TrainingParams()
training_params.__dict__

{'batch_size': 16,
 'eval_strategy': 'epoch',
 'learning_rate': 2e-05,
 'num_train_epochs': 3,
 'weight_decay': 0.01,
 'warmup_steps': 100}

In [12]:
training_params.batch_size = 32
training_params.__dict__

{'batch_size': 32,
 'eval_strategy': 'epoch',
 'learning_rate': 2e-05,
 'num_train_epochs': 3,
 'weight_decay': 0.01,
 'warmup_steps': 100}

In [13]:
trained_ner_model, model_out_path = train.train_model(model, dataset, label_list, training_params, tokenized_dataset, tokenizer)

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.132866,0.645961,0.645242,0.645601,0.958499
2,0.209300,0.114102,0.662823,0.719811,0.690143,0.962463
3,0.105700,0.113090,0.674068,0.734836,0.703142,0.963925


## 4. Optimization

### Evaluation and optimization

* Evaluation: Stratified k-fold Cross-Validation - one of ["None", k] (https://huggingface.co/docs/datasets/v1.11.0/splits.html#examples; https://discuss.huggingface.co/t/k-fold-cross-validation/5765/5, https://github.com/huggingface/datasets/discussions/5940)
* Optimization:
  * https://huggingface.co/docs/transformers/hpo_train
  * https://medium.com/carbon-consulting/transformer-models-hyperparameter-optimization-with-the-optuna-299e185044a8
  * Grid Search (GridSearchCV?), Line Search, hyperopt
* Analyze grid search results using Heatmap or similar

### Optimization

For optimization (hyperparameter search) we use the HF Trainer API (https://huggingface.co/docs/transformers/hpo_train#hyperparameter-search-using-trainer-api) and optuna (https://optuna.readthedocs.io/en/stable/index.html, `pip install optuna`). We can access all of the search parameters, as they are stored in a dict structure, and also adapt the number of trials `n_trials` to run. 

In [ ]:
optimize_params = config.OptimizeParams()

optimize_params.hp_space["learning_rate"]["param_type"]
optimize_params.n_trials = 10
optimize_params.__dict__

In [ ]:
#to remove an hyperparameter from the hp_space dict, use pop() or similar
optimize_params.hp_space.pop("warmup_steps")
optimize_params.__dict__

In [ ]:
model_out_path = "roberta_germeval14_opt/"
eval_opt.optimize(optimize_params, training_params, model.path, model_out_path, label_list, tokenizer, tokenized_dataset)

## 5. Evaluation

In [14]:
class_report, errors = eval_opt.compute_metrics_per_tag(trained_ner_model, tokenized_dataset, label_list) 

              precision    recall  f1-score   support

         LOC       0.77      0.78      0.77      1708
         ORG       0.51      0.54      0.52      1025
         PER       0.69      0.76      0.72       899

   micro avg       0.67      0.71      0.69      3632
   macro avg       0.66      0.69      0.67      3632
weighted avg       0.68      0.71      0.69      3632



In [15]:
eval_opt.save_class_report(class_report, "md", model_out_path)

In [16]:
config.save_train_config(model_out_path, training_params)

In [36]:
#inference example
from transformers import pipeline, AutoModelForTokenClassification

text = "Novelle	von Joh. L. Buchta."

id2label = {0:label_list[0], 1:label_list[1], 2:label_list[2], 3:label_list[3], 4:label_list[4], 5:label_list[5], 6:label_list[6]}
finetuned_model = AutoModelForTokenClassification.from_pretrained("xlm-roberta-base_hisgermaner_2025-04-02_14-45/checkpoint-276", num_labels=7, id2label=id2label)
classifier = pipeline("ner", model=finetuned_model, tokenizer=tokenizer)
classifier(text)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


[{'entity': 'B-PER',
  'score': np.float32(0.8944226),
  'index': 4,
  'word': '▁Joh',
  'start': 12,
  'end': 15},
 {'entity': 'I-PER',
  'score': np.float32(0.7188401),
  'index': 5,
  'word': '.',
  'start': 15,
  'end': 16},
 {'entity': 'I-PER',
  'score': np.float32(0.95051396),
  'index': 6,
  'word': '▁L',
  'start': 17,
  'end': 18},
 {'entity': 'I-PER',
  'score': np.float32(0.9569103),
  'index': 7,
  'word': '.',
  'start': 18,
  'end': 19},
 {'entity': 'I-PER',
  'score': np.float32(0.9494691),
  'index': 8,
  'word': '▁Buch',
  'start': 20,
  'end': 24},
 {'entity': 'I-PER',
  'score': np.float32(0.95551276),
  'index': 9,
  'word': 'ta',
  'start': 24,
  'end': 26}]